# Determinant via Divide-and-Conquer (Schur Complement)

The standard cofactor expansion computes $\det(M)$ in $O(n!)$. We can do much better by splitting the matrix into quadrants and using the **Schur complement**.

Given a $2n \times 2n$ matrix partitioned as:

$$M = \begin{pmatrix} A & B \\ C & D \end{pmatrix}$$

If $A$ is invertible, then:

$$\det(M) = \det(A) \cdot \det(D - C A^{-1} B)$$

Instead of computing $A^{-1}$ explicitly, we use $A^{-1} = \frac{1}{\det(A)} \text{adj}(A)$, giving:

$$\det(M) = \frac{\det(D \cdot \det(A) - C \cdot \text{adj}(A) \cdot B)}{\det(A)^{n-1}}$$

We recurse on both the top-left block and the Schur complement. For odd-sized matrices, we pad with a 1 on the diagonal (which doesn't change the determinant).

## Implementation

Three components:
- **`cofactor_matrix`** — computes the classical cofactor matrix (used for $\text{adj}(A) = C^T$)
- **`augment_matrix_with_zeros`** — pads odd-sized matrices to even size by adding a row/column with a 1 in the corner
- **`determinant`** — the recursive divide-and-conquer: split into $A, B, C, D$, compute $\det(A)$, form the Schur complement $D \cdot a - C \cdot \text{adj}(A) \cdot B$, and recurse

The cofactor step is $O(n^3)$ per level, and we halve the problem each time, giving roughly $O(n^3 \log n)$ overall (the title's $n^{\log n}$ refers to the number of recursive calls, not the total arithmetic cost).

## Helper: Generate a Test Matrix

We generate a random invertible matrix to verify our implementation against NumPy's `linalg.det`.

In [2]:
import numpy as np

def generate_invertible_matrix(size):
    """Generates a random invertible matrix of a given size."""
    while True:
        # Generate a random matrix with elements drawn from a uniform distribution
        matrix = np.random.rand(size, size)
        # Check if the matrix is invertible by calculating its determinant
        if np.linalg.det(matrix) != 0:
            return matrix

# Example usage
size = 3
invertible_matrix = generate_invertible_matrix(size)
print(invertible_matrix)

[[0.69438503 0.32764064 0.13643726]
 [0.27657606 0.54644365 0.61568481]
 [0.46157821 0.00937576 0.35027203]]


In [3]:
import numpy as np

def cofactor_matrix(matrix):
    """Calculate the cofactor matrix."""
    n, m = matrix.shape
    cofactors = np.zeros((n, m))

    for i in range(n):
        for j in range(m):
            # Create the minor matrix by removing row i and column j
            minor_matrix = np.delete(np.delete(matrix, i, axis=0), j, axis=1)
            # Calculate the cofactor
            cofactor = ((-1) ** (i + j)) * np.linalg.det(minor_matrix)
            cofactors[i, j] = cofactor

    return cofactors

def augment_matrix_with_zeros(matrix):
    """Augment the matrix by adding a row and column of zeros, with the last element as 1."""
    n, m = matrix.shape
    # Create a new matrix with an extra row and column filled with zeros
    augmented_matrix = np.zeros((n + 1, m + 1))

    # Copy the original matrix into the top-left corner of the new matrix
    augmented_matrix[:n, :m] = matrix

    # Set the last element to 1
    augmented_matrix[n, m] = 1

    return augmented_matrix

def determinant(matrix):
    """Calculate the determinant of a square matrix."""
    n, m = matrix.shape

    if n == 1:
        return matrix[0, 0]
    elif n == 2:
        return matrix[0, 0] * matrix[1, 1] - matrix[0, 1] * matrix[1, 0]
    else:
        # If the matrix size is odd, augment it with zeros to make it even
        if n % 2 == 1:
            matrix = augment_matrix_with_zeros(matrix)

        row, col = matrix.shape
        row2, col2 = row // 2, col // 2

        # Split the matrix into four submatrices
        A = matrix[:row2, :col2]
        B = matrix[:row2, col2:]
        C = matrix[row2:, :col2]
        D = matrix[row2:, col2:]

        a = determinant(A)
        adj = cofactor_matrix(A).T
        return determinant(D * a - C @ (adj @ B)) / (a ** (row2 - 1))

# Example usage
print("Determinant (Custom):", determinant(invertible_matrix))
print("Determinant (NumPy):", np.linalg.det(invertible_matrix))


Determinant (Custom): 0.15621069920251804
Determinant (NumPy): 0.15621069920251804
